In [ ]:
import numpy
import sys
import nrrd
from scipy import ndimage
import tomllib
import os
import meshio

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import export_centerline


with open("../src/config.toml", mode="rb") as f:
    config = tomllib.load(f)

sequence = "T2"

if "T2":
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file_T2"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename_T2"]
    segment_folder  = config["data"]["segment_folder_T2"]
    out_folder  = config["data"]["centerline_folder"]
    mesh_folder = config["data"]["output_folder"]
else:
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename"]
    segment_folder  = config["data"]["segment_folder"]
    out_folder  = config["data"]["centerline_folder"]
    mesh_folder = config["data"]["output_folder"]



## IDs

In [31]:
eyeball_id = 1
lens_id = 6

h = 0.9

## Load segments, pixel spacing etc

In [ ]:
data, header = nrrd.read(base + segment_folder + segment_filename)
pixel_spacing = numpy.loadtxt( input_file[:-4] + "_pixel_spacing.txt")

in_folder = base + mesh_folder + "re-meshed_h_"+str(h).replace(".","-") +"_cenetered_"+sequence + "/"

lens_mesh_filename = in_folder + "id_"+str(lens_id)+"__h_"+str(h).replace(".","-")+".stl"  # your .inp mesh file
eyeball_mesh_filename = in_folder + "id_"+str(eyeball_id)+"_shell__h_"+str(h).replace(".","-")+".stl"  # your .inp mesh file

mesh_lens = meshio.read(lens_mesh_filename)
mesh_eyeball = meshio.read(eyeball_mesh_filename)



## Lense axis

In [33]:


lense_center = mesh_lens.points.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh_lens.points - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction = vh[-1]  # lens: first axis

def axis(t):
    return lense_center + t * axis_direction


axis_points_visual = [axis(-20.0), axis(20.0)]

# export5centerline(axis_points_visual, 1.0 , base + out_folder + "lense_axis.vtp", [0,0,0])



lense_center =  [ 4.86500593 -6.60888113 -0.93641105]


## Length

In [34]:
P = mesh_eyeball.points
v = P - lense_center
t = v @ axis_direction                                              # position along the axis
r = numpy.linalg.norm(v - numpy.outer(t, axis_direction), axis=1)   # distance from the axis

tol = 5                                                 
near = r < tol
if near.sum() < 2:
    raise ValueError("No mesh points close to the axis -> increase tol")

t_near = t[near]
axial_length = t_near.max() - t_near.min()
print("axial length =", axial_length)

end_points = [axis(t_near.min()), axis(t_near.max())]          # for visualization
if sequence=="T2":
    export_centerline(end_points, 1.0, base + out_folder + "axial_length_T2.vtp", [0, 0, 0])
else:
    export_centerline(end_points, 1.0, base + out_folder + "axial_length.vtp", [0, 0, 0])

axial length = 24.025567868889702
